In [2]:

import numpy as np
import jax.numpy as jnp
from jax.scipy.linalg import expm
from jax.numpy import kron
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from jax import random
from matplotlib.collections import LineCollection
import matplotlib.patches as patches
from matplotlib.colors import Normalize
import pandas as pd
from scipy.linalg import expm
import matplotlib.colors as mcolors
import jax
import re
import ast
import itertools
from scipy.sparse import coo_matrix
import os
from itertools import count
from pathlib import Path
jax.config.update("jax_enable_x64", True)
d_min=0.5



In [3]:
import os

SAVING = False
NOTEBOOK_NAME = "default"
_fig_counter = 1  # מונה גלובלי לכל הפונקציות

def set_notebook_name(name):
    global NOTEBOOK_NAME, _fig_counter
    NOTEBOOK_NAME = name
    _fig_counter = 1  # איפוס מונה

def maybe_save(name):
    if SAVING:
        global _fig_counter
        
        params = f"_G{Gamma:.2f}_R{R:.1f}_J{Jmax:.1f}_N{N}"
        fig_name = f"Fig{_fig_counter}_{name}{params}"
        _fig_counter += 1
        
        save_dir = f'figures/{NOTEBOOK_NAME}'
        os.makedirs(save_dir, exist_ok=True)
        filepath = f'{save_dir}/{fig_name}.pdf'
        plt.savefig(filepath, bbox_inches='tight', dpi=300)
        print(f"Saved: {filepath}")

In [4]:
def dipolar_J(distance_nm, theta_deg=0):
    """
    Calculate dipolar coupling J for two electron spins separated by `distance_nm` nanometers.
    
    Parameters:
        distance_nm : float
            Distance between spins in nanometers.
        theta_deg : float
            Angle (degrees) between inter-spin vector and quantization axis (default 0).
    
    Returns:
        J in Hz
    """
    # Physical constants
    mu0 = 4 * np.pi * 1e-7      # vacuum permeability (H/m)
    muB = 9.2740100783e-24      # Bohr magneton (J/T)
    h = 6.62607015e-34          # Planck constant (J*s)
    g_e = 2.00231930436256      # g-factor for electron

    # Convert distance to meters
    r_m = distance_nm * 1e-9
    theta_rad = np.deg2rad(theta_deg)

    # Dipolar coupling formula (electron spins)
    J_Hz = (mu0 / (4 * np.pi)) * (g_e * muB)**2 / (h * r_m**3) * (1 - 3 * np.cos(theta_rad)**2)

    return J_Hz

In [5]:

# Define the integrands
def I1(k):
    return quad(lambda x: np.exp(-x) / x**3, k, np.inf, limit=200)[0]

def I2(k):
    return quad(lambda x: np.exp(-x) / x**6, k, np.inf, limit=200)[0]

def numerator_integrals(a):
    I1 = quad(lambda u: u**(-1.5) * np.exp(-u), a, np.inf, limit=200)[0]
    I2 = quad(lambda u: u**(-3) * np.exp(-u), a, np.inf, limit=200)[0]
    return np.sqrt(I2 - I1**2), I1
def installing():
    print(sys.executable)
    !{sys.executable} -m pip install jax jaxlib



In [6]:
def lindbladmatpl(H, Gamma, exclude_site=None, double_precision=False):
    dtype = jnp.complex128 if double_precision else jnp.complex64
    N = H.shape[0]
    Hmat1 = kron(H, jnp.eye(H.shape[0], dtype=dtype))
    Hmat2 = kron(jnp.eye(H.shape[0], dtype=dtype), H.T)
    mat = -1j*(Hmat1 - Hmat2)
    for k in range(N):
        if k == exclude_site:
            continue
        ket = jnp.zeros(N, dtype=dtype).at[k].set(1.)
        L = jnp.outer(ket, ket)
        LL = L.T @ L
        LrhoL1 = jnp.repeat(jnp.repeat(L, N, axis=0), N, axis=1)
        LrhoL2 = jnp.tile(L.T, (N,N))
        LrhoL = LrhoL1 * LrhoL2
        LLrho = jnp.repeat(jnp.repeat(LL, N, axis=0), N, axis=1) * jnp.tile(jnp.eye(N, dtype=dtype), (N,N))
        rhoLL = jnp.tile(LL.T, (N,N)) * jnp.repeat(jnp.repeat(jnp.eye(N, dtype=dtype), N, axis=0), N, axis=1)
        mat += Gamma * (LrhoL - 0.5 * (LLrho + rhoLL))
    return mat

In [7]:
def random_bath_positions(key, radius, num_particles, dim=2, d_min=0.5, max_tries=1000):

    if dim not in [2, 3]:
        raise ValueError("dim must be 2 or 3")

    positions = []
    tries = 0
    key_main = key

    while len(positions) < num_particles:
        tries += 1
        if tries > max_tries:
            raise RuntimeError(f"Could not place all points after {max_tries} tries — try reducing num_particles or radius.")

        key_main, key_r, key_dir = random.split(key_main, 3)

        # רדיוסים אחידים לפי נפח
        r = radius * (random.uniform(key_r, shape=(1,)) ** (1.0 / dim))

        # כיוונים אחידים על מעגל / כדור
        if dim == 2:
            theta = 2 * jnp.pi * random.uniform(key_dir, shape=(1,))
            x = r * jnp.cos(theta)
            y = r * jnp.sin(theta)
            candidate = jnp.concatenate([x, y])
        else:  # dim == 3
            u = random.uniform(key_dir, shape=(1,))
            key_main, key_phi = random.split(key_main)
            v = random.uniform(key_phi, shape=(1,))
            theta = 2 * jnp.pi * u
            phi = jnp.arccos(2 * v - 1)
            x = r * jnp.sin(phi) * jnp.cos(theta)
            y = r * jnp.sin(phi) * jnp.sin(theta)
            z = r * jnp.cos(phi)
            candidate = jnp.concatenate([x, y, z])

        # בדיקה למרחק מינימלי (>= 1 ביחידות מנורמלות)
        if all(jnp.linalg.norm(candidate - p) >= 1.0 for p in positions):
            positions.append(candidate)

    return jnp.stack(positions)


In [8]:
def Hampl_geometric(key, positions, Jmax, alpha, omega=0.0, double_precision=False):
    dtype = jnp.complex128 if double_precision else jnp.complex64
    N = positions.shape[0]
    key, subkey = random.split(key)

    ref_index = random.randint(subkey, (), 0, N)
    ref_pos = positions[ref_index]

    dists_to_ref = jnp.linalg.norm(positions - ref_pos, axis=1)
    sort_order = jnp.argsort(dists_to_ref)
    sorted_pos = positions[sort_order]

    X = sorted_pos[:, None, :]
    Y = sorted_pos[None, :, :]
    dist_matrix = jnp.linalg.norm(X - Y, axis=-1)

    Jij = jnp.where(dist_matrix != 0, Jmax / dist_matrix**alpha, 0.0)
    Jij = jnp.array(Jij, dtype=dtype)  # ← FIX: Force correct dtype

    H0 = omega * jnp.eye(N, dtype=dtype)

    H = H0 + Jij
    return H, sort_order
####################
def Hampl_geometric(key, positions, Jmax, alpha, omega=0.0, double_precision=False):
    dtype = jnp.complex128 if double_precision else jnp.complex64
    N = positions.shape[0]

    # --- deterministic reference: extreme right site ---
    ref_index = jnp.argmax(positions[:, 0])   # x-max
    ref_pos = positions[ref_index]

    dists_to_ref = jnp.linalg.norm(positions - ref_pos, axis=1)
    sort_order = jnp.argsort(dists_to_ref)
    sorted_pos = positions[sort_order]

    X = sorted_pos[:, None, :]
    Y = sorted_pos[None, :, :]
    dist_matrix = jnp.linalg.norm(X - Y, axis=-1)

    Jij = jnp.where(dist_matrix != 0, Jmax / dist_matrix**alpha, 0.0)
    Jij = jnp.array(Jij, dtype=dtype)

    H0 = omega * jnp.eye(N, dtype=dtype)
    H = H0 + Jij

    return H, sort_order


In [9]:

def evolve_rho_over_time(L, rho0, tlist, double_precision=False):
    if double_precision:
        L = L.astype(jnp.complex128)
        rho0 = rho0.astype(jnp.complex128)
    
    N = rho0.shape[0]
    rho0_vec = rho0.flatten()

    evals, evecs = np.linalg.eig(np.array(L))
    Vinv = np.linalg.inv(evecs)

    coeffs0 = Vinv @ rho0_vec

    rhos = []
    for t in tlist:
        lam_real = evals.real
        lam_imag = evals.imag
        lam_imag_mod = np.remainder(lam_imag * t, 2*np.pi)
        evals_mod = lam_real * t + 1j * lam_imag_mod
        exp_factor = np.exp(evals_mod)
        rho_vec_t = evecs @ (exp_factor * coeffs0)
        rhos.append(rho_vec_t.reshape((N, N)))

    return np.stack(rhos)


def animate_rho_heatmap(rhos, tlist, interval=500, vmax=None):
   n = rhos[0].shape[0]
   fig, ax = plt.subplots(figsize=(4, 4))
   im = ax.imshow(jnp.abs(rhos[0]), cmap='viridis', vmin=0, vmax=vmax)
   title = ax.set_title(f"t = {tlist[0]:.2f}")
   plt.colorbar(im, ax=ax)
   
   def update(frame):
       im.set_data(jnp.abs(rhos[frame]))
       title.set_text(f"t = {tlist[frame]:.2f}")
       return im, title
   
   anim = FuncAnimation(fig, update, frames=len(tlist), interval=interval, blit=False)
   plt.close(fig)
   return HTML(anim.to_jshtml())

In [ ]:
def plot_rho_geometric_frame(rho, positions, coherence_percentile=80, title=None, radius=None, point_scale=100):
    N = rho.shape[0]
    D = positions.shape[1]
    rho_abs = jnp.abs(rho)
    populations = jnp.real(jnp.diag(rho))
    
    mask = ~jnp.eye(N, dtype=bool)
    threshold = jnp.percentile(rho_abs[mask], 100 - coherence_percentile)
    
    # ✅ Normalize positions for plotting if radius given
    if radius is not None:
        positions = positions / radius
        radius = 1.0

    fig = plt.figure(figsize=(6, 6))
    
    if D == 3:
        ax = fig.add_subplot(projection='3d')
        for i in range(N):
            ax.scatter(positions[i,0], positions[i,1], positions[i,2], 
                       s=300 * populations[i], c=populations[i], cmap='viridis', vmin=0, vmax=1)
        
        for i in range(N):
            for j in range(i + 1, N):
                if rho_abs[i, j] >= threshold:
                    coords = jnp.stack([positions[i], positions[j]])
                    ax.plot(*coords.T, color='gray', alpha=0.5)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        
    else:  # 2D
        ax = fig.add_subplot()
        
        if radius is not None:
            circle = patches.Circle((0, 0), radius, edgecolor='red', facecolor='none',
                                    linestyle='--', linewidth=2, zorder=0)
            ax.add_patch(circle)
            ax.set_xlim(-radius * 1.05, radius * 1.05)
            ax.set_ylim(-radius * 1.05, radius * 1.05)
        
        scatter_points = []
        for i in range(N):
            sc = ax.scatter(positions[i,0], positions[i,1], s=point_scale, 
                            c=populations[i], cmap='viridis', vmin=0, vmax=1, zorder=3)
            scatter_points.append(sc)
        
        lines = []
        strengths = []
        for i in range(N):
            for j in range(i + 1, N):
                if rho_abs[i, j] >= threshold:
                    lines.append([positions[i], positions[j]])
                    strengths.append(rho_abs[i, j])
        
        if lines:
            norm = Normalize(vmin=0, vmax=1)
            lc = LineCollection(lines, cmap='viridis', array=np.array(strengths), norm=norm,
                                linewidths=2, alpha=0.8, zorder=2)
            ax.add_collection(lc)
        
        ax.set_aspect('equal')
        ax.set_xticks([]); ax.set_yticks([])
    
    if title:
        ax.set_title(title)
    
    plt.tight_layout()
    maybe_save("rho_geometric") 
    plt.show()


In [ ]:
def animate_rho_heatmap_and_geometry(rhos, tlist, positions, coherence_percentile=80, radius=None, interval=500, point_scale=100):
    N = rhos[0].shape[0]
    vmax = 1
    vmin = 0

    # ✅ Normalize positions for plotting if radius given
    if radius is not None:
        positions = positions / radius
        radius = 1.0

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

    # Initial heatmap
    im = ax1.imshow(jnp.abs(rhos[0]), cmap='viridis', vmin=vmin, vmax=vmax)
    cb1 = plt.colorbar(im, ax=ax1)
    ax1.set_title(f"ρ(t) heatmap, t = {tlist[0]:.2f}")
    ax1.set_xticks([]); ax1.set_yticks([])

    # Initial geometric plot
    rho_abs = jnp.abs(rhos[0])
    populations = jnp.real(jnp.diag(rhos[0]))

    if radius is not None:
        circle = patches.Circle((0, 0), radius, edgecolor='red', facecolor='none',
                                linestyle='--', linewidth=2, zorder=0)
        ax2.add_patch(circle)
        ax2.set_xlim(-radius * 1.05, radius * 1.05)
        ax2.set_ylim(-radius * 1.05, radius * 1.05)

    for i in range(N):
        ax2.scatter(positions[i,0], positions[i,1], s=point_scale, 
                    c=populations[i], cmap='viridis', vmin=vmin, vmax=vmax, zorder=3)

    mask = ~jnp.eye(N, dtype=bool)
    threshold = jnp.percentile(rho_abs[mask], 100 - coherence_percentile)
    lines, strengths = [], []
    for i in range(N):
        for j in range(i + 1, N):
            if rho_abs[i, j] >= threshold:
                lines.append([positions[i], positions[j]])
                strengths.append(rho_abs[i, j])

    if lines:
        lc = LineCollection(lines, cmap='viridis', array=np.array(strengths),
                            linewidths=2, alpha=0.8, zorder=2)
        ax2.add_collection(lc)

    ax2.set_aspect('equal')
    ax2.set_xticks([]); ax2.set_yticks([])

    def update(frame):
        ax1.clear()
        ax2.clear()

        rho = rhos[frame]

        # Update heatmap
        im = ax1.imshow(jnp.abs(rho), cmap='viridis', vmin=vmin, vmax=vmax)
        ax1.set_title(f"ρ(t) heatmap, t = {tlist[frame]:.2f}")
        ax1.set_xticks([]); ax1.set_yticks([])

        # Update geometric plot
        rho_abs = jnp.abs(rho)
        populations = jnp.real(jnp.diag(rho))

        if radius is not None:
            circle = patches.Circle((0, 0), radius, edgecolor='red', facecolor='none',
                                    linestyle='--', linewidth=2, zorder=0)
            ax2.add_patch(circle)
            ax2.set_xlim(-radius * 1.05, radius * 1.05)
            ax2.set_ylim(-radius * 1.05, radius * 1.05)

        for i in range(N):
            ax2.scatter(positions[i,0], positions[i,1], s=point_scale, 
                        c=populations[i], cmap='viridis', vmin=vmin, vmax=vmax, zorder=3)

        threshold = jnp.percentile(rho_abs[mask], 100 - coherence_percentile)
        lines, strengths = [], []
        for i in range(N):
            for j in range(i + 1, N):
                if rho_abs[i, j] >= threshold:
                    lines.append([positions[i], positions[j]])
                    strengths.append(rho_abs[i, j])

        if lines:
            lc = LineCollection(lines, cmap='viridis', array=np.array(strengths), norm=Normalize(vmin=vmin, vmax=vmax),
                                linewidths=2, alpha=0.8, zorder=2)
            ax2.add_collection(lc)

        ax2.set_aspect('equal')
        ax2.set_xticks([]); ax2.set_yticks([])

        return im,

    anim = FuncAnimation(fig, update, frames=len(tlist), interval=interval, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
def analyze_rho_evolution(rho_list, H=None, L=None, times=None, tol=1e-12):
    """
    Analyze a list of density matrices rho(t) for physical consistency.
    Also checks unitarity for H (using stable diagonalization + 2π folding)
    and dissipativity for L.
    
    *** UPDATED: Now preserves double precision throughout calculation ***
    """
    results = {
        "hermiticity_err": [],
        "trace_dev": [],
        "min_eig": [],
        "purity": [],
        "energy": [],
        "unitarity_err": [],       # ||U† U - I|| if H is given
        "max_real_eig_L": []       # max Re(λ(L)) if L is given
    }

    N = rho_list[0].shape[0]
    
    # *** FIX: Detect and preserve dtype ***
    input_dtype = rho_list[0].dtype
    np_dtype = np.complex128 if input_dtype == jnp.complex128 else np.complex64
    
    # --- Unitarity check for H ---
    if H is not None:
        # *** FIX: Preserve H's dtype ***
        H_np = np.array(H, dtype=np_dtype)
        
        # Diagonalize H once (more stable than repeated expm for large times)
        evals_H, evecs_H = np.linalg.eig(H_np)
        Vinv_H = np.linalg.inv(evecs_H)

        for t in (times if times is not None else range(len(rho_list))):
            # Separate real (decay-like) and imaginary (oscillatory) parts of λ
            lam_real = evals_H.real
            lam_imag = evals_H.imag

            # Fold the oscillatory part λ_imag * t into [-π, π] range
            lam_imag_mod = np.remainder(lam_imag * t, 2*np.pi)

            # Recombine: keep decay part unchanged, fold only oscillations
            evals_mod = lam_real * t + 1j * lam_imag_mod

            # Construct U(t) in a numerically stable way
            U = evecs_H @ np.diag(np.exp(-1j * evals_mod)) @ Vinv_H

            # Compute unitarity error
            err = np.linalg.norm(U.conj().T @ U - np.eye(N, dtype=np_dtype))
            results["unitarity_err"].append(err)

    # --- Dissipativity check for L ---
    if L is not None:
        # *** FIX: Preserve L's dtype ***
        L_np = np.array(L, dtype=np_dtype)
        
        # Maximum real part of the Liouvillian spectrum
        evals = np.linalg.eigvals(L_np)
        max_real = np.max(evals.real)
        # This is constant if L is fixed in time
        results["max_real_eig_L"] = [max_real] * len(rho_list)

    # --- General rho(t) checks ---
    for rho in rho_list:
        # *** FIX: Preserve rho's dtype ***
        rho_np = np.array(rho, dtype=np_dtype)
        
        herm_err = np.linalg.norm(rho_np - rho_np.conj().T)   # Hermiticity
        trace_dev = np.abs(np.trace(rho_np) - 1)              # Trace ~ 1
        min_eig = np.min(np.linalg.eigvalsh(rho_np))          # Positivity check
        purity = np.trace(rho_np @ rho_np).real               # Purity
        
        # Energy calculation
        if H is not None:
            energy = np.trace(H_np @ rho_np).real
        else:
            energy = np.nan
            
        results["hermiticity_err"].append(herm_err)
        results["trace_dev"].append(trace_dev)
        results["min_eig"].append(min_eig)
        results["purity"].append(purity)
        results["energy"].append(energy)

    df = pd.DataFrame(results)

    # --- Print summary ---
    if H is not None:
        print(f"Max unitarity error for H: {np.max(df['unitarity_err']):.2e}")
    if L is not None:
        print(f"Max real part of eigenvalues(L): {df['max_real_eig_L'][0]:.4e}")

    return df

def plot_rho_analysis(df, times):
    """
    Plot rho analysis metrics over time, including unitarity and Liouvillian checks.
    """
    nrows = 6 if "unitarity_err" in df and df["unitarity_err"].notna().any() else 4
    fig, axes = plt.subplots(nrows, 1, figsize=(8, 2.5*nrows), sharex=True)

    axes[0].plot(times, df["hermiticity_err"], label="Hermiticity Error")
    axes[0].set_ylabel("||ρ - ρ†||"); axes[0].set_yscale("log"); axes[0].legend()

    axes[1].plot(times, df["trace_dev"], label="Trace Deviation")
    axes[1].set_ylabel("|Tr(ρ) - 1|"); axes[1].set_yscale("log"); axes[1].legend()

    axes[2].plot(times, df["min_eig"], label="Min Eigenvalue")
    axes[2].set_ylabel("Min λ(ρ)"); axes[2].legend()

    axes[3].plot(times, df["purity"], label="Purity Tr(ρ²)")
    axes[3].set_ylabel("Purity"); axes[3].legend()

    row = 4
    if "unitarity_err" in df and df["unitarity_err"].notna().any():
        axes[row].plot(times, df["unitarity_err"], label="Unitarity Error ||U†U - I||")
        axes[row].set_ylabel("Unitarity Error"); axes[row].set_yscale("log"); axes[row].legend()
        row += 1

    if "max_real_eig_L" in df and df["max_real_eig_L"].notna().any():
        axes[row].plot(times, df["max_real_eig_L"], label="Max Re[λ(L)]")
        axes[row].axhline(0, color='r', linestyle='--', alpha=0.6)
        axes[row].set_ylabel("Max Re eigenvalue(L)"); axes[row].legend()

    axes[-1].set_xlabel("Time")
    plt.tight_layout()
    maybe_save("rho_analysis") 
    plt.show()

In [ ]:
def plot_site0_population(rhos, tlist):
    populations = [jnp.real(rho[0,0]) for rho in rhos]
    plt.figure(figsize=(8, 5))
    plt.plot(tlist, populations, 'b-', linewidth=2)
    plt.xlabel('Time')
    plt.ylabel('⟨0|ρ(t)|0⟩')
    plt.title('Population of Site 0 vs Time')
    plt.grid(True, alpha=0.3)
    maybe_save("site0_population") 
    plt.show()

In [ ]:
def pairwise_J(positions, Jmax, alpha):
    X = positions[:, None, :]; Y = positions[None, :, :]
    d = jnp.linalg.norm(X - Y, axis=-1) + jnp.eye(len(positions))  # avoid /0 on diag
    J = Jmax / d**alpha
    return J * (1 - jnp.eye(len(positions)))  # zero diagonal

def J_characteristic(J, method="mean_top_k", k=3, pct=95):
    Js = np.sort(np.array(J).ravel())[::-1]
    Js = Js[Js > 0]
    if method == "max":        return float(Js[0])
    if method == "mean":       return float(Js.mean())
    if method == "mean_top_k": return float(Js[:min(k, len(Js))].mean())
    if method == "percentile": return float(np.percentile(Js, pct))
    raise ValueError("method ∈ {'max','mean','mean_top_k','percentile'}")

In [ ]:
def histogram_P0_over_realizations(
    num_realizations=100, N=8, R=25.0, d_min=0.5,
    Jmax=1.0, alpha=3.0, tlist=50.0,  # יכול להיות float או list/ndarray
    Jchar_method="mean_top_k", k=3, pct=95,
    dim=2, keys_in=None, seed=None
):
    rng = np.random.default_rng(seed)
    keys_out = []

    # ודא ש-tlist הוא iterable
    if np.isscalar(tlist):
        tlist = [float(tlist)]
    else:
        tlist = list(map(float, tlist))

    # פלטים
    P0_fixed_all = np.zeros((len(tlist), num_realizations))
    P0_norm_all = np.zeros((len(tlist), num_realizations))
    Jchar_list = np.zeros(num_realizations)

    # יצירת keys
    if keys_in is None:
        keys_in = [random.PRNGKey(int(rng.integers(0, 2**31-1))) for _ in range(num_realizations)]

    for idx in range(num_realizations):
        key = keys_in[idx]
        keys_out.append(key)

        # --- Geometry ---
        R_units = R / d_min
        positions = random_bath_positions(key, radius=R_units, num_particles=N, dim=dim, d_min=1.0)
        H, order = Hampl_geometric(key, positions, Jmax=Jmax, alpha=alpha, omega=0.0)
        Jij = pairwise_J(positions, Jmax, alpha)

        # --- Jchar ---
        Js = np.sort(np.array(Jij).ravel())[::-1]
        Js = Js[Js > 0]
        if Jchar_method == "max":
            Jchar = float(Js[0])
        elif Jchar_method == "mean":
            Jchar = float(Js.mean())
        elif Jchar_method == "mean_top_k":
            Jchar = float(Js[:min(k, len(Js))].mean())
        elif Jchar_method == "percentile":
            Jchar = float(np.percentile(Js, pct))
        else:
            raise ValueError("Invalid Jchar_method")
        Jchar_list[idx] = Jchar

        Gamma = 0.1 * Jmax
        L = lindbladmatpl(H, Gamma)
        rho0 = jnp.zeros((N, N)).at[0, 0].set(1.0)

        # --- Loop over times ---
        for ti, t_phys in enumerate(tlist):
            t_norm = t_phys / max(Jchar, 1e-12)
            rhos = evolve_rho_over_time(L, rho0, jnp.array([t_phys, t_norm]))
            P0_fixed_all[ti, idx] = float(jnp.real(rhos[0][0, 0]))
            P0_norm_all[ti, idx] = float(jnp.real(rhos[1][0, 0]))

    # החזרת מבנה גמיש — וקטור אם רק זמן אחד
    if len(tlist) == 1:
        return {
            "P0_fixed": P0_fixed_all[0],
            "P0_norm": P0_norm_all[0],
            "t_fixed": tlist[0],
            "t_norm": tlist[0] / Jchar_list,
            "Jchar": Jchar_list,
            "keys_out": keys_out
        }
    else:
        return {
            "P0_fixed": P0_fixed_all,  # shape: (len(tlist), num_realizations)
            "P0_norm": P0_norm_all,
            "tlist": np.array(tlist),
            "Jchar": Jchar_list,
            "keys_out": keys_out
        }


In [ ]:
def plot_P0_histograms(res, bins=40):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.hist(res["P0_fixed"], bins=bins, alpha=0.9)
    plt.title(f"P0 at fixed time t={res['meta']['t_phys']}")
    plt.xlabel("P0"); plt.ylabel("count")
    plt.subplot(1,2,2); plt.hist(res["P0_norm"], bins=bins, alpha=0.9)
    plt.title(f"P0 at normalized time τ={res['meta']['tau']} (norm={res['meta']['Jchar_method']})")
    plt.xlabel("P0")
    plt.tight_layout(); 
    maybe_save("P0_histograms")  
    plt.show()

In [ ]:
def animate_histogram_from_histogramP0(res, bins=30, interval=400):
    """
    Create an animation from the output of histogram_P0_over_realizations
    when called with a tlist of multiple times.

    Parameters
    ----------
    res : dict
        Output of histogram_P0_over_realizations with multi-time tlist.
    bins : int
        Number of bins for histograms.
    interval : int
        Interval between frames in milliseconds.

    Returns
    -------
    HTML animation object (for Jupyter) and FuncAnimation object.
    """
    P0_fixed = res["P0_fixed"]   # shape: (len(tlist), num_realizations)
    P0_norm  = res["P0_norm"]
    tlist    = res["tlist"]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    def update(frame_idx):
        axes[0].clear()
        axes[1].clear()

        axes[0].hist(P0_fixed[frame_idx], bins=bins, range=(0, 1))
        axes[0].set_title(f"P0 Fixed Time t={tlist[frame_idx]:.2f}")
        axes[0].set_xlim(0, 1)

        axes[1].hist(P0_norm[frame_idx], bins=bins, range=(0, 1))
        axes[1].set_title(f"P0 Normalized Time t={tlist[frame_idx]:.2f}")
        axes[1].set_xlim(0, 1)

        return axes

    anim = FuncAnimation(fig, update, frames=len(tlist), interval=interval, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml()), anim

In [ ]:
def plot_rho_geometric_frame_with_hist(rho, positions, coherence_percentile=80, title=None, radius=None, point_scale=100):
    N = rho.shape[0]
    D = positions.shape[1]
    rho_abs = jnp.abs(rho)
    populations = jnp.real(jnp.diag(rho))

    mask = ~jnp.eye(N, dtype=bool)
    coherences = np.array(rho_abs[mask])
    threshold = np.percentile(coherences, 100 - coherence_percentile)

    # ✅ Normalize positions for plotting if radius given
    if radius is not None:
        positions = positions / radius
        radius = 1.0

    fig, (ax_geom, ax_hist) = plt.subplots(1, 2, figsize=(12, 6))

    # --- Left: geometric plot ---
    if D == 2:
        if radius is not None:
            circle = patches.Circle((0, 0), radius, edgecolor='red', facecolor='none',
                                    linestyle='--', linewidth=2, zorder=0)
            ax_geom.add_patch(circle)
            ax_geom.set_xlim(-radius * 1.05, radius * 1.05)
            ax_geom.set_ylim(-radius * 1.05, radius * 1.05)

        scatter_points = []
        for i in range(N):
            sc = ax_geom.scatter(positions[i,0], positions[i,1], s=point_scale, 
                                 c=populations[i], cmap='viridis', vmin=0, vmax=1, zorder=3)
            scatter_points.append(sc)

        lines = []
        strengths = []
        for i in range(N):
            for j in range(i + 1, N):
                if rho_abs[i, j] >= threshold:
                    lines.append([positions[i], positions[j]])
                    strengths.append(rho_abs[i, j])

        if lines:
            norm = Normalize(vmin=0, vmax=1)
            lc = LineCollection(lines, cmap='viridis', array=np.array(strengths), norm=norm,
                                linewidths=2, alpha=0.8, zorder=2)
            ax_geom.add_collection(lc)
            plt.colorbar(lc, ax=ax_geom, label='|Coherence|')

        ax_geom.set_aspect('equal')
        ax_geom.set_xticks([]); ax_geom.set_yticks([])
        if title:
            ax_geom.set_title(title)

        plt.colorbar(scatter_points[0], ax=ax_geom, label='Population')
    
    else:
        raise NotImplementedError("3D plot not yet supported with histogram layout.")

    # --- Right: histogram ---
    ax_hist.hist(coherences, bins=30, color='skyblue', edgecolor='black')
    ax_hist.set_xlabel("|rho_ij|")
    ax_hist.set_ylabel("Count")
    ax_hist.set_title("Coherence magnitude distribution")
    ax_hist.set_yscale("log")  # אופציונלי: לוג scale כדי לראות סדרי גודל
    

    plt.tight_layout()
    plt.show()


In [ ]:


def animate_rho_geometry_with_hist(rhos, tlist, positions, coherence_percentile=80, radius=None, point_scale=100, bins=30):
    """
    Animation: geometry plot + histogram of all coherences for each frame.
    Based on original animate_rho_heatmap_and_geometry with minimal changes.
    """
    N = rhos[0].shape[0]
    D = positions.shape[1]

    fig, (ax_geom, ax_hist) = plt.subplots(1, 2, figsize=(12, 5))

    def update(frame_idx):
        ax_geom.clear()
        ax_hist.clear()

        rho = rhos[frame_idx]
        rho_abs = np.abs(np.array(rho))
        populations = np.real(np.diag(rho))

        # --- Geometry plot (same as original) ---
        if radius is not None:
            circle = patches.Circle((0, 0), radius, edgecolor='red', facecolor='none',
                                    linestyle='--', linewidth=2, zorder=0)
            ax_geom.add_patch(circle)
            ax_geom.set_xlim(-radius * 1.05, radius * 1.05)
            ax_geom.set_ylim(-radius * 1.05, radius * 1.05)

        sc = ax_geom.scatter(positions[:, 0], positions[:, 1], s=point_scale,
                             c=populations, cmap='viridis', vmin=0, vmax=1, zorder=3)

        mask = ~np.eye(N, dtype=bool)
        coherences_all = rho_abs[mask]  # all off-diagonal values
        threshold = np.percentile(coherences_all, 100 - coherence_percentile)

        lines = []
        strengths = []
        for i in range(N):
            for j in range(i + 1, N):
                if rho_abs[i, j] >= threshold:
                    lines.append([positions[i], positions[j]])
                    strengths.append(rho_abs[i, j])

        if lines:
            norm = Normalize(vmin=0, vmax=1)
            lc = LineCollection(lines, cmap='viridis', array=np.array(strengths),
                                norm=norm, linewidths=2, alpha=0.8, zorder=2)
            ax_geom.add_collection(lc)

        ax_geom.set_aspect('equal')
        ax_geom.set_xticks([]); ax_geom.set_yticks([])
        ax_geom.set_title(f"Geometry at t={tlist[frame_idx]:.2e}")
        plt.colorbar(sc, ax=ax_geom, label='Population')

        # --- Histogram of all coherences ---
        ax_hist.hist(coherences_all, bins=bins, color='skyblue', edgecolor='black')
        ax_hist.set_xlabel("|rho_ij|")
        ax_hist.set_ylabel("Count")
        ax_hist.set_title("All coherences (off-diagonal)")

    anim = FuncAnimation(fig, update, frames=len(tlist), interval=300, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml()), anim


In [ ]:
def plot_p0_and_purity(rhos, tlist):
    """
    Plot P0(t) and global purity Tr[rho(t)^2] on the same figure,
    with a dashed horizontal line at 1/N (fully mixed state limit).
    """
    N = rhos[0].shape[0]
    p0 = [np.real(rho[0, 0]) for rho in rhos]  # אוכלוסיה באתר 0
    purity = [np.real(np.trace(rho @ rho)) for rho in rhos]  # Tr[rho^2]
    
    plt.figure(figsize=(7,5))
    plt.plot(tlist, p0, label=r"$P_0(t)$", color='blue')
    plt.plot(tlist, purity, label=r"Purity $\mathrm{Tr}[\rho^2]$", color='red')
    plt.axhline(1/N, color='gray', linestyle='--', label=r"$1/N$ limit")

    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.title("Local vs Global Thermalization")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    maybe_save("p0_purity") 
    plt.show()


In [ ]:
def animate_rho_heatmap_geometry_hist(rhos, tlist, positions, coherence_percentile=80, radius=None, interval=500, point_scale=100, bins=30):
    N = rhos[0].shape[0]
    vmax = 1
    vmin = 0

    # ✅ Normalize positions for plotting if radius given
    if radius is not None:
        positions = positions / radius
        radius = 1.0

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

    # --- Initial heatmap ---
    im = ax1.imshow(jnp.abs(rhos[0]), cmap='viridis', vmin=vmin, vmax=vmax)
    cb1 = plt.colorbar(im, ax=ax1)
    ax1.set_title(f"ρ(t) heatmap, t = {tlist[0]:.2f}")
    ax1.set_xticks([]); ax1.set_yticks([])

    # --- Initial geometric plot ---
    rho_abs = jnp.abs(rhos[0])
    populations = jnp.real(jnp.diag(rhos[0]))

    if radius is not None:
        circle = patches.Circle((0, 0), radius, edgecolor='red', facecolor='none',
                                linestyle='--', linewidth=2, zorder=0)
        ax2.add_patch(circle)
        ax2.set_xlim(-radius * 1.05, radius * 1.05)
        ax2.set_ylim(-radius * 1.05, radius * 1.05)

    for i in range(N):
        ax2.scatter(positions[i,0], positions[i,1], s=point_scale, 
                    c=populations[i], cmap='viridis', vmin=vmin, vmax=vmax, zorder=3)

    mask = ~jnp.eye(N, dtype=bool)
    threshold = jnp.percentile(rho_abs[mask], 100 - coherence_percentile)
    lines, strengths = [], []
    for i in range(N):
        for j in range(i + 1, N):
            if rho_abs[i, j] >= threshold:
                lines.append([positions[i], positions[j]])
                strengths.append(rho_abs[i, j])

    if lines:
        lc = LineCollection(lines, cmap='viridis', array=np.array(strengths),
                            norm=Normalize(vmin=vmin, vmax=vmax),
                            linewidths=2, alpha=0.8, zorder=2)
        ax2.add_collection(lc)

    ax2.set_aspect('equal')
    ax2.set_xticks([]); ax2.set_yticks([])

    # --- Initial histogram ---
    coherences_all = np.array(rho_abs[mask])
    ax3.hist(coherences_all, bins=bins, color='skyblue', edgecolor='black')
    ax3.set_xlabel("|rho_ij|")
    ax3.set_ylabel("Count")
    ax3.set_title("All coherences")

    def update(frame):
        ax1.clear()
        ax2.clear()
        ax3.clear()

        rho = rhos[frame]

        # Heatmap
        im = ax1.imshow(jnp.abs(rho), cmap='viridis', vmin=vmin, vmax=vmax)
        ax1.set_title(f"ρ(t) heatmap, t = {tlist[frame]:.2f}")
        ax1.set_xticks([]); ax1.set_yticks([])

        # Geometry
        rho_abs = jnp.abs(rho)
        populations = jnp.real(jnp.diag(rho))

        if radius is not None:
            circle = patches.Circle((0, 0), radius, edgecolor='red', facecolor='none',
                                    linestyle='--', linewidth=2, zorder=0)
            ax2.add_patch(circle)
            ax2.set_xlim(-radius * 1.05, radius * 1.05)
            ax2.set_ylim(-radius * 1.05, radius * 1.05)

        for i in range(N):
            ax2.scatter(positions[i,0], positions[i,1], s=point_scale, 
                        c=populations[i], cmap='viridis', vmin=vmin, vmax=vmax, zorder=3)

        threshold = jnp.percentile(rho_abs[mask], 100 - coherence_percentile)
        lines, strengths = [], []
        for i in range(N):
            for j in range(i + 1, N):
                if rho_abs[i, j] >= threshold:
                    lines.append([positions[i], positions[j]])
                    strengths.append(rho_abs[i, j])

        if lines:
            lc = LineCollection(lines, cmap='viridis', array=np.array(strengths),
                                norm=Normalize(vmin=vmin, vmax=vmax),
                                linewidths=2, alpha=0.8, zorder=2)
            ax2.add_collection(lc)

        ax2.set_aspect('equal')
        ax2.set_xticks([]); ax2.set_yticks([])

        # Histogram
        coherences_all = np.array(rho_abs[mask])
        ax3.hist(coherences_all, bins=bins, color='skyblue', edgecolor='black')
        ax3.set_xlabel("|rho_ij|")
        ax3.set_ylabel("Count")
        ax3.set_title("All coherences")

        return im,

    anim = FuncAnimation(fig, update, frames=len(tlist), interval=interval, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
def plot_distance_to_thermal(rhos, tlist):
    """
    Plot two normalized 'distance-to-thermal' metrics on the same axis:
      1) purity_norm(t) = (Tr[rho^2] - 1/N) / (1 - 1/N)  in [0,1]
      2) spread_norm(t) = (ΔP/P)(t) / (ΔP/P)(t=0)         in [0,1]
    Both start near 1 (pure, non-uniform) and tend to 0 as the state thermalizes.
    """
    import numpy as np
    import matplotlib.pyplot as plt

    N = rhos[0].shape[0]
    # Purity
    purity = np.array([np.real(np.trace(r @ r)) for r in rhos])
    purity_norm = (purity - 1.0/N) / (1.0 - 1.0/N)

    # ΔP/P: std(p) / mean(p); here mean(p)=1/N ⇒ ΔP/P = N * std(p)
    pops = np.array([np.real(np.diag(r)) for r in rhos])  # shape: (T, N)
    std_p = pops.std(axis=1, ddof=0)
    spread_rel = N * std_p
    spread_norm = spread_rel / (spread_rel[0] if spread_rel[0] != 0 else 1.0)

    plt.figure(figsize=(7,5))
    plt.plot(tlist, purity_norm, label="Normalized purity gap", lw=2)
    plt.plot(tlist, spread_norm, label="Relative spread ΔP/P (normalized)", lw=2)
    plt.ylim(-0.05, 1.05)
    plt.xlabel("Time")
    plt.ylabel("Distance to thermal (0=thermal)")
    plt.title("Convergence of global and population metrics")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    maybe_save("distance_to_thermal")
    plt.show()


In [ ]:
def plot_decay_curves_with_threshold(L, tlist, threshold=0.2, sort_order=None):
    """
    מציייר העיכות לוגריתמיות לכל מצב התחלתי עם סימון חציית threshold
    ומחזיר טבלה עם זמני החציה.
    """
    N = int(np.sqrt(L.shape[0]))
    
    # אם לא נתן sort_order, השתמש בסדר הטבעי
    if sort_order is None:
        sort_order = np.arange(N)
    
    crossing_times = []
    plt.figure(figsize=(8,6))
    
    # עבור על האתרים לפי הסדר הגיאומטרי
    for plot_idx, start_site in enumerate(sort_order):
        rho0 = np.zeros((N, N))
        rho0[start_site, start_site] = 1.0
        rhos = evolve_rho_over_time(L, rho0, tlist)
        pops = np.array([rho[start_site, start_site].real for rho in rhos])
        # מציאת זמן חציה
        idx_cross = np.where(pops < threshold)[0]
        if len(idx_cross) > 0:
            cross_time = tlist[idx_cross[0]]
            crossing_times.append(cross_time)
            # גרף עם סימון
            plt.semilogy(tlist, pops, label=f"Site {start_site} (geom #{plot_idx})")
            #plt.scatter(cross_time, pops[idx_cross[0]], c='red', marker='x', zorder=5)
        else:
            crossing_times.append(None)
            plt.semilogy(tlist, pops, label=f"Site {start_site} (geom #{plot_idx})")
    
    plt.xlabel("Time")
    plt.ylabel("Population P_i(t)")
    plt.title(f"Decay curves with {threshold} threshold (geometric order)")
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend()
    maybe_save("threshold_simple")  # NEW: Only one line added!
    plt.show()
    
    # טבלת זמני חציה
    print("Crossing times (geometric order):")
    for i, (site_idx, t_cross) in enumerate(zip(sort_order, crossing_times)):
        if t_cross is not None:
            print(f"Site {site_idx} (geom #{i}): {t_cross:.3e}")
        else:
            print(f"Site {site_idx} (geom #{i}): Did not cross threshold")
    
    return crossing_times

In [ ]:


def plot_projection_heatmap(proj_matrix, decay_rates, Gamma=0.1):
    """
    Plot a basic heatmap of |projection| for each site vs Liouvillian decay rate.
    
    Parameters
    ----------
    proj_matrix : ndarray (N_sites, n_modes)
        Raw projection amplitudes (already in sorted basis).
    decay_rates : ndarray (n_modes,)
        Decay rates (-Re eigenvalues of L).
    Gamma : float
        Lindblad parameter for thresholding
    """
    with plt.rc_context({
        "axes.labelsize":  30,
        "xtick.labelsize": 30,
        "ytick.labelsize": 30,
        "legend.fontsize": 30,
    }):
    # Filter only positive decay rates
        upper_thresh = 0.1 * Gamma   
        lower_thresh = 1e-15
        
        mask = (decay_rates > lower_thresh) & (decay_rates < upper_thresh)
        decay_rates_pos = decay_rates[mask]
        proj_matrix_pos = proj_matrix[:, mask]
            
        order_eig_pos = np.argsort(decay_rates_pos)[::-1]  # Fast → slow
        
        # No row sorting needed - data is already in geometric order
        proj_matrix_sorted = proj_matrix_pos[:, order_eig_pos]
        decay_rates_sorted = decay_rates_pos[order_eig_pos]
        
        # Build bin edges for log-scale X axis
        log_gamma = np.log(decay_rates_sorted)
        gamma_edges = np.zeros(len(decay_rates_sorted) + 1)
        gamma_edges[0] = np.exp(log_gamma[0] - (log_gamma[1] - log_gamma[0]) / 2)
        for i in range(1, len(decay_rates_sorted)):
            gamma_edges[i] = np.sqrt(decay_rates_sorted[i-1] * decay_rates_sorted[i])
        gamma_edges[-1] = np.exp(log_gamma[-1] + (log_gamma[-1] - log_gamma[-2]) / 2)
        
        # Site edges (y-axis)
        site_edges = np.arange(proj_matrix_sorted.shape[0] + 1) - 0.5
        
        # Plot
        fig, ax = plt.subplots(figsize=(10, 6))
        X, Y = np.meshgrid(gamma_edges, site_edges)
        im = ax.pcolormesh(X, Y, np.abs(proj_matrix_sorted),
                           norm=mcolors.LogNorm(vmin=np.abs(proj_matrix_sorted).min()+1e-15,
                                                vmax=np.abs(proj_matrix_sorted).max()),
                           cmap='viridis', shading='flat')
        ax.set_xscale('log')
        ax.set_xlabel("Decay rate γ")
        ax.set_ylabel("Site index")
        #ax.set_title("|Projection| onto Liouvillian eigenmodes")
        plt.colorbar(im, ax=ax, label='|Projection|')
        plt.tight_layout()
        maybe_save("projection_heatmap")
        plt.show()

In [ ]:
def plot_projection_bars(proj_matrix, decay_rates, Gamma=0.1):
    import numpy as np
    import matplotlib.pyplot as plt

    gamma_low, gamma_high = 1e-12, 0.1 * Gamma
    mask = (decay_rates > gamma_low) & (decay_rates < gamma_high)
    decay_sel = decay_rates[mask]
    proj_sel = np.abs(proj_matrix[:, mask])**2  # shape: (N_sites, n_selected_modes)

    n_bins = max(1, round(np.log10(gamma_high / gamma_low)))
    bins = np.logspace(np.log10(gamma_low), np.log10(gamma_high), n_bins + 1)
    bin_centers = np.sqrt(bins[:-1] * bins[1:])

    N_sites = proj_sel.shape[0]

    for i in range(N_sites):
        hist_i, _ = np.histogram(decay_sel, bins=bins, weights=proj_sel[i])
        total = hist_i.sum()
        hist_norm = hist_i / total if total > 0 else hist_i

        plt.figure(figsize=(6, 4))
        plt.bar(bin_centers, hist_norm, width=np.diff(bins), align='center', edgecolor='black', color='mediumslateblue')
        plt.xscale('log')
        plt.xlabel(r"Decay rate $\gamma$")
        plt.ylabel("Normalized weight")
        plt.title(f"Site {i} projection onto decay modes")
        plt.grid(True, which="both", ls="--", alpha=0.5)
        plt.tight_layout()
        plt.show()


In [ ]:


def plot_time_evolution_heatmap(L, tlist, sort_order=None, thresholds=[0.5, 0.9]):
    """
    Plot heatmap of population decay over time for each starting site,
    similar style to cumulative projection heatmap.
    
    Parameters:
    -----------
    L : ndarray
        Liouvillian matrix
    tlist : ndarray
        Time points for evolution
    sort_order : ndarray or None
        Optional order of sites (e.g., geometric order). If None, natural order is used.
    thresholds : list
        List of fractional population thresholds to mark
    """
    N = int(np.sqrt(L.shape[0]))  # number of sites
    if sort_order is None:
        sort_order = np.arange(N)
    
    pop_matrix = np.zeros((N, len(tlist)))
    
    # evolve from each site
    for site_idx in range(N):
        rho0 = np.zeros((N, N))
        rho0[site_idx, site_idx] = 1.0
        rhos = evolve_rho_over_time(L, rho0, tlist)
        pop_matrix[site_idx, :] = [rho[site_idx, site_idx].real for rho in rhos]
    
    # reorder rows
    pop_matrix = pop_matrix[sort_order, :]
    
    # bin edges
    time_edges = np.zeros(len(tlist) + 1)
    log_t = np.log(tlist)
    time_edges[0] = np.exp(log_t[0] - (log_t[1] - log_t[0]) / 2)
    for i in range(1, len(tlist)):
        time_edges[i] = np.sqrt(tlist[i-1] * tlist[i])
    time_edges[-1] = np.exp(log_t[-1] + (log_t[-1] - log_t[-2]) / 2)
    
    site_edges = np.arange(N + 1) - 0.5
    
    # plot
    fig, ax = plt.subplots(figsize=(10, 6))
    X, Y = np.meshgrid(time_edges, site_edges)
    im = ax.pcolormesh(X, Y, pop_matrix, cmap='viridis', vmin=0, vmax=1, shading='flat')
    plt.colorbar(im, ax=ax, label='Population')
    
    ax.set_xscale('log')
    ax.set_xlabel("Time")
    ax.set_ylabel("Initial site index (geometric order)")
    ax.set_title("Population decay over time (per initial site)")
    
    # threshold markers
    colors = ['red', 'magenta', 'cyan']
    for i, thr in enumerate(thresholds):
        color = colors[i % len(colors)]
        for row_idx in range(pop_matrix.shape[0]):
            vals = pop_matrix[row_idx, :]
            crossing_indices = np.where(vals <= thr)[0]
            if len(crossing_indices) > 0:
                t_val = tlist[crossing_indices[0]]
                ax.scatter(t_val, row_idx, c=color, s=30, zorder=10,
                           label=f'{thr*100:.0f}% threshold' if row_idx == 0 else "",
                           edgecolors='black', linewidth=0.5)
    
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()
    
    return fig, ax


In [ ]:
def plot_decay_curves_with_threshold_2(L=None, tlist=None, threshold=0.2, 
                                        N=None, M=1,
                                        key=None, positions=None, 
                                        Jmax=None, alpha=None, Gamma=None,
                                        sort_order=None, P_SHOW=True, show_purity=True, 
                                        show_diffusion=True,
                                        show_populations=False, excite_site=0, animate_populations=False,
                                        no_legend=False):  # ✅ new param
    """
    Plot decay curves with LOG X-axis and regular Y-axis, with threshold crossing markers.
    Returns table of crossing times.

    NEW: If M is provided, builds system from scratch for M-excitation subspace.
    OLD: If L is provided, uses original single-excitation behavior.

    NEW FEATURES:
    - show_populations=True: Show all P_i(t) from single excitation
    - animate_populations=True: Video of all P_i(t) for each excitation site
    - no_legend=True: Disable legends in all plots
    """

    if L is None:
        if M == 1:
            H, sort_order_used = Hampl_geometric(key, positions, Jmax=Jmax, alpha=alpha, omega=0.0)
            L_system = lindbladmatpl(H, Gamma, exclude_site=0)
            N_sites = N

            def create_initial_state(start_site):
                rho0 = np.zeros((N_sites, N_sites))
                rho0[start_site, start_site] = 1.0
                return rho0

            def extract_population(rho, site):
                return rho[site, site].real
        else:
            eps = jnp.zeros(N)
            basis, idx_map = build_fixed_M_basis(N, M)
            H, sort_order_used = Hampl_half_exc(key, positions, eps, Jmax, alpha, basis, idx_map, omega=0.0)
            L_system = lindbladmatpl(H, Gamma)
            N_sites = N

            def create_initial_state(start_site):
                rho0_subspace = np.zeros((basis.shape[0], basis.shape[0]), dtype=complex)
                mask_site = (basis[:, start_site] == 1)
                occupied_indices = np.where(mask_site)[0]
                if len(occupied_indices) > 0:
                    weight = 1.0 / len(occupied_indices)
                    rho0_subspace[np.ix_(occupied_indices, occupied_indices)] = weight * np.eye(len(occupied_indices))
                return rho0_subspace

            def extract_population(rho, site):
                rho_site = half_exc_to_site_matrix(rho, basis, idx_map)
                rho_site_sorted = rho_site[np.ix_(sort_order_used, sort_order_used)]
                return rho_site_sorted[site, site].real
    else:
        L_system = L
        N_sites = int(np.sqrt(L.shape[0]))
        sort_order_used = sort_order if sort_order is not None else np.arange(N_sites)

        def create_initial_state(start_site):
            rho0 = np.zeros((N_sites, N_sites))
            rho0[start_site, start_site] = 1.0
            return rho0

        def extract_population(rho, site):
            return rho[site, site].real

    if sort_order_used is None:
        sort_order_used = np.arange(N_sites)

    crossing_times = []
    all_rhos = {} if (show_diffusion or show_populations or animate_populations) else None

    # === THRESHOLD PLOT ===
    if P_SHOW:
        plt.figure(figsize=(8,6))
        for plot_idx, start_site in enumerate(sort_order_used):
            start_site = int(start_site)
            rho0 = create_initial_state(start_site)
            rhos = evolve_rho_over_time(L_system, rho0, tlist)
            pops = np.array([extract_population(rho, start_site) for rho in rhos])

            if show_diffusion or show_populations or animate_populations:
                all_rhos[start_site] = rhos

            plt.semilogx(tlist, pops, label=f"Site {start_site}")
        plt.xlabel("Time")
        plt.ylabel(r"Site population $P_i(t)$")
        plt.title(f"Decay curves (M={M} excitation)")
        plt.grid(True, which="both", ls="--", alpha=0.5)
        if not no_legend:
            plt.legend()
        maybe_save("threshold")
        plt.show()
    else:
        for plot_idx, start_site in enumerate(sort_order_used):
            start_site = int(start_site)
            rho0 = create_initial_state(start_site)
            rhos = evolve_rho_over_time(L_system, rho0, tlist)
            if show_diffusion or show_populations or animate_populations:
                all_rhos[start_site] = rhos

    # === ANIMATION ===
    if animate_populations:
        from matplotlib.animation import FuncAnimation
        from IPython.display import HTML, display

        fig, ax = plt.subplots(figsize=(8,6))

        def update_frame(frame_idx):
            ax.clear()
            excite_site_actual = int(sort_order_used[frame_idx])
            rhos = all_rhos[excite_site_actual]

            for plot_idx, start_site in enumerate(sort_order_used):
                start_site = int(start_site)
                pops = np.array([extract_population(rho, start_site) for rho in rhos])
                ax.semilogx(tlist, pops, label=rf"$P_{{{start_site}}}(t)$")

            ax.set_xlabel("Time")
            ax.set_ylabel(r"Site population $P_i(t)$")
            ax.set_title(f"All populations from excitation at site {excite_site_actual} (M={M})")
            ax.grid(True, which="both", ls="--", alpha=0.5)
            if not no_legend:
                ax.legend()

        anim = FuncAnimation(fig, update_frame, frames=len(sort_order_used), interval=1000, blit=False)
        plt.close(fig)

        animation_html = HTML(anim.to_jshtml())
        display(animation_html)

    # === PURITY PLOT ===
    for plot_idx, start_site in enumerate(sort_order_used):
        start_site = int(start_site)
        if all_rhos is None or start_site not in all_rhos:
            rho0 = create_initial_state(start_site)
            rhos = evolve_rho_over_time(L_system, rho0, tlist)
            if all_rhos is None:
                all_rhos = {}
            all_rhos[start_site] = rhos

    plt.figure(figsize=(8,6))
    for plot_idx, start_site in enumerate(sort_order_used):
        start_site = int(start_site)
        if L is None and M > 1:
            purity = []
            for rho in all_rhos[start_site]:
                rho_site = half_exc_to_site_matrix(rho, basis, idx_map)
                purity.append(np.real(np.trace(rho_site @ rho_site)))
            purity = np.array(purity)
        else:
            purity = np.array([np.real(np.trace(rho @ rho)) for rho in all_rhos[start_site]])

        idx_cross = np.where(purity < threshold)[0]
        if len(idx_cross) > 0:
            cross_time = tlist[idx_cross[0]]
            crossing_times.append(cross_time)
            plt.semilogx(tlist, purity, label=f"Site {start_site}")
            #plt.scatter(cross_time, purity[idx_cross[0]], c='red', marker='x', zorder=5)
        else:
            crossing_times.append(None)
            plt.semilogx(tlist, purity, label=f"Site {start_site}")

    plt.xlabel("Time")
    plt.ylabel("Purity Tr(ρ²)")
    plt.title(f"Global thermalization with {threshold} threshold (geometric order)")
    plt.grid(True, which="both", ls="--", alpha=0.5)
    if not no_legend:
        plt.legend()
    maybe_save("purity")
    plt.show()

    # === DIFFUSION PLOT ===
    if show_diffusion and positions is not None and (L is not None or M == 1):
        plt.figure(figsize=(8,6))
        for plot_idx, start_site in enumerate(sort_order_used):
            start_site = int(start_site)
            rhos = all_rhos[start_site]
            start_pos = positions[start_site]
            distances_sq = np.array([np.linalg.norm(pos - start_pos)**2 for pos in positions])

            msd = []
            for rho in rhos:
                pops = np.real(np.diag(rho))
                msd.append(np.sum(distances_sq * pops))
            msd = np.array(msd)

            dt = np.diff(tlist)
            dmsd_dt = np.diff(msd)
            D_t = dmsd_dt / dt
            t_mid = (tlist[1:] + tlist[:-1]) / 2

            plt.loglog(t_mid, D_t, label=f"Site {start_site}")

        plt.xlabel("Time")
        plt.ylabel("D(t) = d⟨R²⟩/dt")
        plt.title("Time-dependent diffusion (geometric order)")
        plt.grid(True, which="both", ls="--", alpha=0.5)
        if not no_legend:
            plt.legend()
        maybe_save("diffusion")
        plt.show()

    # === SINGLE POPULATION PLOT ===
    if show_populations:
        excite_sites = excite_site if isinstance(excite_site, (list, tuple, np.ndarray)) else [excite_site]
        for excite_site_actual in excite_sites:
            excite_site_actual = int(sort_order_used[excite_site_actual])
            if excite_site_actual in all_rhos:
                rhos = all_rhos[excite_site_actual]
            else:
                rho0 = create_initial_state(excite_site_actual)
                rhos = evolve_rho_over_time(L_system, rho0, tlist)
    
            plt.figure(figsize=(8,6))
            for plot_idx, start_site in enumerate(sort_order_used):
                start_site = int(start_site)
                pops = np.array([extract_population(rho, start_site) for rho in rhos])
                plt.semilogx(tlist, pops, label=rf"$P_{{{start_site}}}(t)$")

            plt.xlabel("Time")
            plt.ylabel( r"Site population $P_i(t)$")
            plt.title(f"All populations from excitation at site {excite_site_actual} (M={M})")
            plt.grid(True, which="both", ls="--", alpha=0.5)
            if not no_legend:
                plt.legend()
            maybe_save(f"populations_site{excite_site_actual}")
            plt.show()

    # === PRINT CROSSING TIMES ===
    print("Crossing times (geometric order):")
    for i, (site_idx, t_cross) in enumerate(zip(sort_order_used, crossing_times)):
        site_idx = int(site_idx)
        if t_cross is not None:
            print(f"Site {site_idx} : {t_cross:.3e}")
        else:
            print(f"Site {site_idx}: Did not cross threshold")

    return crossing_times


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-


def extract_functions_from_file(input_file, output_file):
    """
    מחלץ רק פונקציות מקובץ פייתון ושומר אותן בקובץ נפרד
    """
    
    with open(input_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    # חילוץ imports
    imports = []
    lines = content.split('\n')
    
    for line in lines:
        stripped = line.strip()
        if (stripped.startswith('import ') or 
            stripped.startswith('from ') or
            stripped.startswith('#') and ('coding' in stripped or 'env python' in stripped)):
            imports.append(line)
    
    # ניתוח הקוד עם AST לחילוץ פונקציות
    try:
        tree = ast.parse(content)
    except SyntaxError as e:
        print(f"שגיאת syntax: {e}")
        return False
    
    functions = []
    
    class FunctionExtractor(ast.NodeVisitor):
        def __init__(self, source_lines):
            self.source_lines = source_lines
        
        def visit_FunctionDef(self, node):
            # חילוץ הפונקציה מהקוד המקורי
            start_line = node.lineno - 1  # AST מתחיל מ-1, רשימות מ-0
            
            # מציאת סוף הפונקציה
            if hasattr(node, 'end_lineno') and node.end_lineno:
                end_line = node.end_lineno
            else:
                # חישוב גס של סוף הפונקציה
                end_line = start_line + 1
                base_indent = len(self.source_lines[start_line]) - len(self.source_lines[start_line].lstrip())
                
                for i in range(start_line + 1, len(self.source_lines)):
                    line = self.source_lines[i]
                    if line.strip() == "":
                        continue
                    current_indent = len(line) - len(line.lstrip())
                    if current_indent <= base_indent and line.strip():
                        break
                    end_line = i + 1
            
            # חילוץ הפונקציה
            func_lines = self.source_lines[start_line:end_line]
            functions.append('\n'.join(func_lines))
            
            self.generic_visit(node)
    
    source_lines = content.split('\n')
    extractor = FunctionExtractor(source_lines)
    extractor.visit(tree)
    
    # יצירת הקובץ החדש
    with open(output_file, "w", encoding="utf-8") as f:
        # כתיבת imports
        f.write('\n'.join(imports))
        f.write('\n\n')
        
        # כתיבת פונקציות
        for i, func in enumerate(functions):
            if i > 0:
                f.write('\n\n')
            f.write(func)
        
        f.write('\n')
    
    print(f"חולצו {len(functions)} פונקציות לקובץ {output_file}")
    return True

def simple_function_extractor(input_file, output_file):
    """
    גרסה פשוטה יותר - מחלצת פונקציות באמצעות regex
    """
    with open(input_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    # חילוץ imports
    import_pattern = r'^(import\s+.+|from\s+.+\s+import\s+.+|#!/.+|#.*coding[:=]\s*.+)$'
    imports = []
    
    lines = content.split('\n')
    for line in lines[:50]:  # בודק רק את 50 השורות הראשונות
        if re.match(import_pattern, line.strip()):
            imports.append(line)
    
    # חילוץ פונקציות
    func_pattern = r'^def\s+\w+\s*\([^)]*\)\s*:'
    functions = []
    current_func = []
    in_function = False
    base_indent = 0
    
    for line in lines:
        # התחלת פונקציה
        if re.match(func_pattern, line):
            if current_func:  # שמירת הפונקציה הקודמת
                functions.append('\n'.join(current_func))
            current_func = [line]
            in_function = True
            base_indent = len(line) - len(line.lstrip())
            continue
        
        if in_function:
            # בדיקה אם הפונקציה נגמרת
            if line.strip():  # שורה לא ריקה
                current_indent = len(line) - len(line.lstrip())
                if current_indent <= base_indent and not line.startswith(' ') and not line.startswith('\t'):
                    # הפונקציה נגמרה
                    functions.append('\n'.join(current_func))
                    current_func = []
                    in_function = False
                    
                    # בדיקה אם זו התחלת פונקציה חדשה
                    if re.match(func_pattern, line):
                        current_func = [line]
                        in_function = True
                        base_indent = len(line) - len(line.lstrip())
                    continue
            
            current_func.append(line)
    
    # שמירת הפונקציה האחרונה
    if current_func:
        functions.append('\n'.join(current_func))
    
    # כתיבת הקובץ
    with open(output_file, "w", encoding="utf-8") as f:
        # imports
        for imp in imports:
            f.write(imp + '\n')
        f.write('\n')
        
        # פונקציות
        for func in functions:
            f.write(func + '\n\n')
    
    print(f"חולצו {len(functions)} פונקציות לקובץ {output_file}")



In [ ]:
def build_fixed_M_basis(N, M):
    """Build all bitstrings length N with exactly M ones."""
    combs = list(itertools.combinations(range(N), M))
    basis_states = jnp.array([
        jnp.array([1 if i in c else 0 for i in range(N)], dtype=int)
        for c in combs
    ])
    state_mapping = {tuple(state.tolist()): i for i, state in enumerate(basis_states)}
    return basis_states, state_mapping

In [ ]:
def build_H_fixed_M_vectorized(basis, idx_map, J):
    """
    Build Hamiltonian in fixed-M subspace using JAX arrays.
    basis: (dim, N) binary occupancy states (jnp.array)
    idx_map: dict mapping state tuple -> index
    J: (N, N) coupling matrix (jnp.array or np.array)
    """
    import itertools
    from scipy.sparse import coo_matrix
    import numpy as np  # still for sparse structure creation

    basis = jnp.asarray(basis)
    J = jnp.asarray(J)
    dim, N = basis.shape

    rows, cols, data = [], [], []

    for i in range(N):
        for j in range(i+1, N):
            Jij = float(J[i, j])
            if Jij == 0.0:
                continue

            # mask for states with (1,0) at (i,j)
            mask_10 = (basis[:, i] == 1) & (basis[:, j] == 0)
            idx_src = jnp.where(mask_10)[0]

            if idx_src.size > 0:
                # create flipped states using JAX ops
                new_states = basis[idx_src].at[:, i].set(0)
                new_states = new_states.at[:, j].set(1)

                # map new states to destination indices
                dst_idx = [idx_map[tuple(state.tolist())] for state in np.array(new_states)]

                # add symmetric entries
                rows.extend(np.array(idx_src))
                cols.extend(dst_idx)
                data.extend([Jij] * len(dst_idx))

                rows.extend(dst_idx)
                cols.extend(np.array(idx_src))
                data.extend([Jij] * len(dst_idx))

    H = coo_matrix((data, (rows, cols)), shape=(dim, dim), dtype=float).tocsr()
    return H


In [ ]:
def reduced_rho_NV(rho_full, basis):
    """Return 2x2 reduced density matrix for NV (site 0) from full rho."""
    mask_1 = (basis[:, 0] == 1)   # indices with NV=1
    mask_0 = ~mask_1               # indices with NV=0

    rho11 = rho_full[np.ix_(mask_1, mask_1)]
    rho10 = rho_full[np.ix_(mask_1, mask_0)]
    rho01 = rho_full[np.ix_(mask_0, mask_1)]
    rho00 = rho_full[np.ix_(mask_0, mask_0)]

    return np.array([
        [np.trace(rho11), np.trace(rho10)],
        [np.trace(rho01), np.trace(rho00)]
    ], dtype=complex)

In [ ]:
def Hampl_geometric_both(key, positions, eps, Jmax, alpha, omega=0.0):
    """
    Build both single-excitation and half-excitation Hamiltonians
    with the same geometric sort_order.
    """
    N = positions.shape[0]

    # --- Single excitation ---
    basis_single, idx_map_single = build_fixed_M_basis(N, M=1)
    H_single, sort_order = Hampl_geometric(key, positions, Jmax, alpha, omega)

    # --- Half excitation (same sort_order) ---
    M_half = N // 2
    basis_half, idx_map_half = build_fixed_M_basis(N, M_half)
    H_half, _ = Hampl_half_exc(
        key, positions, eps, Jmax, alpha,
        basis_half, idx_map_half, omega=omega, sort_order=sort_order
    )

    return (H_single, basis_single, idx_map_single,
            H_half, basis_half, idx_map_half,
            sort_order)

In [ ]:
def Hampl_half_exc(key, positions, eps, Jmax, alpha, basis, idx_map, omega=0.0, sort_order=None):
    """
    Half-excitation version of Hampl_geometric.
    Identical sorting logic, but works in fixed-M excitation subspace.
    """
    N = positions.shape[0]

    # Use provided sort_order or make one
    if sort_order is None:
        key, subkey = random.split(key)
        ref_index = random.randint(subkey, (), 0, N)
        ref_pos = positions[ref_index]
        dists_to_ref = jnp.linalg.norm(positions - ref_pos, axis=1)
        sort_order = jnp.argsort(dists_to_ref)

    sorted_pos = positions[sort_order]

    # Distance matrix
    X = sorted_pos[:, None, :]
    Y = sorted_pos[None, :, :]
    dist_matrix = jnp.linalg.norm(X - Y, axis=-1)

    # Couplings
    Jij = jnp.where(dist_matrix != 0, Jmax / dist_matrix**alpha, 0.0)

    # Project onsite energies into subspace
    sorted_eps = eps[sort_order]
    remapped_basis = basis[:, sort_order]
    onsite_energies = (remapped_basis * sorted_eps).sum(axis=1)
    H0 = jnp.diag(onsite_energies) + omega * jnp.eye(basis.shape[0])

    # Rebuild idx_map for sorted basis
    remapped_idx_map = {tuple(state.tolist()): i for i, state in enumerate(remapped_basis)}

    # XY term
    H1 = build_H_fixed_M_vectorized(remapped_basis, remapped_idx_map, np.array(Jij))

    return H0 + jnp.array(H1.toarray()), sort_order

In [ ]:
def half_exc_to_site_matrix(rho_half, basis, idx_map):
    """
    Map rho in half-filling subspace to an N x N effective 'site-space' matrix.
    Compatible with plotting/analysis functions from the single-excitation code.

    Parameters
    ----------
    rho_half : array_like (dim, dim)
        Density matrix in the fixed-M subspace.
    basis : ndarray (dim, N)
        Binary occupancy states for the fixed-M subspace.
    idx_map : dict
        Mapping from basis tuple to its index (from build_fixed_M_basis).

    Returns
    -------
    site_rho : ndarray (N, N)
        Effective site-space density matrix with populations (diag) and
        coherences (off-diag).
    """
    basis = np.array(basis)
    rho = np.array(rho_half)
    dim, N = basis.shape

    site_rho = np.zeros((N, N), dtype=complex)

    # --- Populations ---
    diag_rho = np.real(np.diag(rho))
    for i in range(N):
        site_rho[i, i] = diag_rho[basis[:, i] == 1].sum()

    # --- Coherences ---
    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            coh_sum = 0.0 + 0.0j
            # states with (i=0, j=1)
            mask_01 = (basis[:, i] == 0) & (basis[:, j] == 1)
            src_indices = np.where(mask_01)[0]
            for src in src_indices:
                target_state = basis[src].copy()
                target_state[i] = 1
                target_state[j] = 0
                dst = idx_map[tuple(target_state.tolist())]
                coh_sum += rho[src, dst]
            site_rho[i, j] = coh_sum

    return site_rho


In [ ]:
def signed_log(y):
    return np.sign(y) * np.log10(np.abs(y))

In [ ]:
def plot_geometry_colored(
        positions, 
        radius=None, 
        point_scale=120,
        title=None,
        cmap_list=None):
    """
    Minimal-change version of plot_rho_geometric_frame,
    but WITHOUT rho(t). Only plots geometry, with fixed site-colors.

    Parameters
    ----------
    positions : (N,2) array
        XY positions (already sorted by your code)
    radius : float or None
        Optional radius to draw dashed circle
    point_scale : int
        Scatter size
    title : str
        Plot title
    cmap_list : optional list of colors
        If None → default matplotlib line-cycle colors
    """

    import matplotlib.pyplot as plt
    from matplotlib import patches
    import numpy as np

    N = positions.shape[0]

    # --- color cycle (same as your line plots: site 0=blue,1=orange,... ) ---
    if cmap_list is None:
        cmap_list = plt.rcParams['axes.prop_cycle'].by_key()['color']

    fig, ax = plt.subplots(figsize=(6, 6))

    # --- radius circle (same as your original implementation) ---
    if radius is not None:
        circle = patches.Circle((0, 0), radius,
                                edgecolor='red', facecolor='none',
                                linestyle='--', linewidth=2, zorder=0)
        ax.add_patch(circle)
        ax.set_xlim(-radius * 1.05, radius * 1.05)
        ax.set_ylim(-radius * 1.05, radius * 1.05)

    # --- main scatter: each site has its color ---
    for i in range(N):
        ax.scatter(
            positions[i, 0], positions[i, 1],
            s=point_scale,
            color=cmap_list[i % len(cmap_list)],
            edgecolors='black',
            linewidth=0.6,
            zorder=3,
            label=f"Site {i}"
        )

    # --- axes formatting (identical to original) ---
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

    if title:
        ax.set_title(title)

    # --- legend placed outside so geometry stays clean ---
    ax.legend(loc='upper right', bbox_to_anchor=(1.20, 1))

    plt.tight_layout()
    plt.show()



def plot_geometry_colored(
        positions, 
        radius=None, 
        point_scale=120,
        title=None,
        cmap_list=None,
        ax=None,
        save_label=None,      # <-- new
        save_dir="figures/paper _graphs"  # <-- optional
    ):
    """
    Supports external ax (for inset).
    If save_label is not None -> saves figure as PDF.
    """
    import matplotlib.pyplot as plt
    from matplotlib import patches
    import numpy as np
    import os

    N = positions.shape[0]

    if cmap_list is None:
        cmap_list = plt.rcParams['axes.prop_cycle'].by_key()['color']

    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
        created_fig = True
    else:
        fig = ax.figure

    # --- radius circle ---
    if radius is not None:
        circle = patches.Circle((0, 0), radius,
                                edgecolor='red', facecolor='none',
                                linestyle='--', linewidth=2, zorder=0)
        ax.add_patch(circle)
        ax.set_xlim(-radius * 1.05, radius * 1.05)
        ax.set_ylim(-radius * 1.05, radius * 1.05)

    # --- scatter ---
    for i in range(N):
        ax.scatter(
            positions[i, 0], positions[i, 1],
            s=point_scale,
            color=cmap_list[i % len(cmap_list)],
            edgecolors='black',
            linewidth=0.6,
            zorder=3,
            label=f"Site {i}"
        )

    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

    if title:
        ax.set_title(title)

    if created_fig:
        ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
        plt.tight_layout()

    # --- SAVE OPTION ---
    if save_label is not None:
        os.makedirs(save_dir, exist_ok=True)
        filepath = f"{save_dir}/{save_label}.pdf"
        fig.savefig(filepath, bbox_inches="tight")
        print(f"Saved: {filepath}")

    if created_fig:
        plt.show()

    return ax


In [ ]:
def animate_T_matrix_vs_gamma(H=None, fixed_seed=5, N=10, threshold=1e-14, R=25.0, Jmax=1.0, alpha=3.0,
                               gamma_list=None, d_min=0.5, double_precision=True,
                               interval=1000, skip_animation=False):
    """
    Animate how the full T_matrix changes with Gamma for a fixed geometry.
    
    Parameters
    ----------
    fixed_seed : int
        Seed for fixed geometry
    interval : int
        Animation interval in ms
    skip_animation : bool
        If True, skip all animation/plotting code and only return the data dictionary.
        This significantly speeds up computation when you only need the matrices.
    """
    # ---------------------
    # signed log transform
    # ---------------------
    def signed_log(y):
        return np.sign(y) * np.log10(1 + np.abs(y))
    
    if gamma_list is None:
        gamma_list = [10**-i for i in np.arange(0, 7, 0.25)]
    gamma_list = np.array(gamma_list)
    R_units = R / d_min
    
    # Generate fixed geometry once
    # Generate geometry OR use external H
    if H is None:
        key = random.PRNGKey(fixed_seed)
        positions = random_bath_positions(key, radius=R_units, num_particles=N, dim=2)
        H, sort_order = Hampl_geometric(key, positions, Jmax, alpha, 
                                       double_precision=double_precision)
    else:
        # if user provides H, keep it
        N = H.shape[0]
        sort_order = None
        positions = None
    
    # Compute T_matrix for all Gamma values
    T_matrices = []
    for gamma in gamma_list:
        L = lindbladmatpl(H, gamma, double_precision=double_precision)
        evals, right_vecs = np.linalg.eig(np.array(L))
        left_vecs = np.linalg.inv(right_vecs)
        N_sites = N
        T_matrix = np.zeros((N_sites, N_sites), dtype=complex)
        
        for k in range(len(evals)):
            lam = evals[k]
            if np.abs(np.real(lam)) < threshold:
                continue
            Lk = left_vecs[k, :]
            Rk = right_vecs[:, k]
            
            for i in range(N_sites):
                rho0 = np.zeros((N_sites, N_sites), dtype=complex)
                rho0[i, i] = 1.0
                proj_L = np.vdot(Lk, rho0.flatten())
                
                for j in range(N_sites):
                    rho_obs = np.zeros((N_sites, N_sites), dtype=complex)
                    rho_obs[j, j] = 1.0
                    proj_R = np.vdot(rho_obs.flatten(), Rk)
                    wk = proj_L * proj_R
                    T_matrix[i, j] += wk / lam
        
        T_matrices.append(np.real(-T_matrix))
    
    # Prepare data dictionary
    data = {
        'gamma_list': gamma_list,
        'T_matrices': T_matrices,
        'positions': positions,
        'sort_order': sort_order
    }
    
    # ========================================
    # EARLY RETURN if skip_animation is True
    # ========================================
    if skip_animation:
        return None, data
    
    # ----------------------------------------
    # Animation code (only runs if needed)
    # ----------------------------------------
    T_matrices_log = [signed_log(T) for T in T_matrices]
    
    # global symmetric color scale
    all_vals = np.concatenate([T.flatten() for T in T_matrices_log])
    vabs = np.max(np.abs(all_vals))
    vmin, vmax = -vabs, vabs
    
    # Animation
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(T_matrices_log[0], origin='lower',
                   cmap='seismic', vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax, label=r"$\mathrm{signed}\,\log_{10}(1+|T_{ij}|)$")
    ax.set_xlabel("Target site j")
    ax.set_ylabel("Source site i")
    title = ax.set_title(f"Spectral transfer matrix (signed-log)\n$\\Gamma$ = {gamma_list[0]:.2e}")
    
    def update(frame):
        im.set_data(T_matrices_log[frame])
        title.set_text(f"Spectral transfer matrix (signed-log)\n$\\Gamma$ = {gamma_list[frame]:.2e}")
        return im, title
    
    anim = FuncAnimation(fig, update, frames=len(gamma_list), interval=interval, blit=False)
    plt.tight_layout()
    plt.close(fig)
    
    # Add T_matrices_log to data when animation is generated
    data['T_matrices_log'] = T_matrices_log
    
    return HTML(anim.to_jshtml()), data

In [ ]:
def gamma_to_latex(gamma, tol=1e-12, digits=2):
    mant, exp = f"{gamma:.12e}".split("e")
    mant = float(mant)
    exp = int(exp)

    if abs(mant - 1.0) < tol:
        return rf"10^{{{exp}}}"
    else:
        mant_str = f"{mant:.{digits}f}".rstrip("0").rstrip(".")
        return rf"{mant_str}\times 10^{{{exp}}}"
        
def animate_populations_from_rhos(
        rhos_list,
        gamma_list,
        tlist,
        sites=None,
        logy=False,
        relative_signed_log=False,
        interval=800):
    """
    Animate P_i(t) from precomputed rhos_list for different Gamma values.

    Parameters
    ----------
    rhos_list : list of arrays
        Each entry is a tensor [T, N, N] representing rho(t) for a specific Gamma.
    gamma_list : list or array
        Gamma values aligned with rhos_list.
    tlist : array
        Time vector (usually geomspace → log scale needed).
    sites : None or list[int]
        Which sites to plot. If None → plot all.
    logy : bool
        Use logarithmic Y-axis on raw populations (ignored if relative_signed_log=True).
    relative_signed_log : bool
        If True → plot signed log distance from equilibrium: 
        f = sign(P - 1/N) * log(1 + |P - 1/N|).
        This avoids divergence at 0 and reveals relaxation structure clearly.
    interval : int
        Frame interval (ms).
    """

    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML

    # Number of sites inferred from the rho tensor
    N = rhos_list[0].shape[1]

    # Default: show all sites
    if sites is None:
        sites = list(range(N))

    # Signed-log transform (stable)
    def signed_log_transform(x):
        return np.sign(x) * np.log(1 + np.abs(x))

    fig, ax = plt.subplots(figsize=(8, 5))

    
    def update(frame):
        ax.clear()

        gamma = gamma_list[frame]
        exp = int(np.round(np.log10(gamma)))
        ax.text(
            0.03, 0.65,
            rf"$\Gamma = {gamma_to_latex(gamma_list[frame])}$",
            transform=ax.transAxes,
            fontsize=18,
            va="top"
        )

        rhos = rhos_list[frame]                # shape: [T, N, N]
        P = np.real(np.diagonal(rhos, axis1=1, axis2=2))  # populations P_i(t)

        if relative_signed_log:
            # Compute signed-log deviation from equilibrium
            P_eq = 1.0 / N
            delta = P - P_eq
            Y = signed_log_transform(delta)
        else:
            # Raw populations
            Y = P

        # Plot selected sites
        for i in sites:
            ax.plot(tlist, Y[:, i], label=f"Site {i}")

        # Logarithmic X-axis always (because tlist is usually geomspace)
        ax.set_xscale('log')

        # Y-axis log only for raw populations
        if logy and not relative_signed_log:
            ax.set_yscale('log')

        # Axis labels
        if relative_signed_log:
            ax.set_ylabel("signed log |P_i(t) - 1/N|")
        else:
            ax.set_ylabel("Population P_i(t)")

        ax.set_xlabel("Time t (log scale)")
        #ax.set_title(f"Population dynamics — Gamma = {gamma_list[frame]:.2e}")
        ax.legend()
        ax.grid(True, which='both', alpha=0.3)

        return ax

        gamma_text = ax.text(
        0.03, 0.95, "",
        transform=ax.transAxes,
        fontsize=18,
        va='top',
        ha='left'
        )

    anim = FuncAnimation(fig, update, frames=len(rhos_list),
                         interval=interval, blit=False)

    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
def simulate_rhos_for_gammas(
        H,
        gamma_list,
        tlist,
        excite_site=0,
        double_precision=True):
    """
    Computes rho(t) for each Gamma in gamma_list.
    Returns a list where each entry is an array [T, N, N].
    """

    import numpy as np
    import jax.numpy as jnp

    N = H.shape[0]

    # Initial state: single excitation at excite_site
    rho0 = jnp.zeros((N, N), dtype=jnp.complex128).at[excite_site, excite_site].set(1.0)

    rhos_list = []

    for Gamma in gamma_list:
        L = lindbladmatpl(H, Gamma, double_precision=double_precision)
        rhos = evolve_rho_over_time(L, rho0, tlist, double_precision=double_precision)
        rhos_list.append(rhos)

    return rhos_list

In [ ]:
def build_H_family(H, target_site, site_list, factor_list):
    """
    Build a list of Hamiltonians H(f), each created by modifying the couplings
    between target_site and the sites in site_list by a factor f.
    
    Parameters
    ----------
    H : array (NxN)
        Original Hamiltonian.
    target_site : int
        Site whose couplings are modified.
    site_list : list[int]
        Sites whose coupling with target_site is scaled.
    factor_list : array-like
        Values of f to apply.
    
    Returns
    -------
    H_list : list[array]
        A list of Hamiltonians, one per factor.
    """

    import numpy as np
    H_list = []

    for f in factor_list:
        H_new = H.copy()
        for j in site_list:
            val = H[target_site, j]
            H_new = H_new.at[target_site, j].set(val * f)
            H_new = H_new.at[j, target_site].set(val * f)
        H_list.append(H_new)

    return H_list


In [ ]:
def animate_Tii_T0i_over_H(Tmats_list, gamma_list, interval=800):
    """
    Create an animation where each frame displays BOTH graphs:
    1) T_ii vs Gamma
    2) T_0i vs Gamma
    for a different Hamiltonian H.

    Parameters
    ----------
    Tmats_list : list of lists
        Each element is T_matrices for a different H
        e.g. Tmats_list[k] = data['T_matrices'] for H_k
    gamma_list : array-like
        Same gamma_list used for all H
    interval : int
        Animation speed (ms)

    Returns
    -------
    HTML animation object
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML

    # Extract sizes
    N = Tmats_list[0][0].shape[0]
    gamma_list = np.array(gamma_list)
    x_vals = np.log10(gamma_list)

    def signed_log(y):
        return np.sign(y) * np.log10(1 + np.abs(y))

    # Prepare figure: two subplots (T_ii above, T_0i below)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10))
    plt.tight_layout()

    def update(frame):
        ax1.clear()
        ax2.clear()

        T_matrices = Tmats_list[frame]

        # ---- T_ii plot ----
        for i in range(N):
            Tii = [T[i, i] for T in T_matrices]
            y_vals = signed_log(Tii)
            ax1.plot(x_vals, y_vals, 'o-', label=f"T[{i},{i}]")

        ax1.set_xlabel(r'$\log_{10}(\Gamma)$')
        ax1.set_ylabel("signed log(T_ii)")
        ax1.set_title(f"T_ii vs Gamma  (H index = {frame})")
        ax1.grid(True, which="both", alpha=0.5)
        ax1.legend(fontsize=8, ncol=2)

        # ---- T_0i plot ----
        for i in range(N):
            T0i = [T[0, i] for T in T_matrices]
            y_vals = signed_log(T0i)
            ax2.plot(x_vals, y_vals, 'o-', label=f"T[0,{i}]")

        ax2.set_xlabel(r'$\log_{10}(\Gamma)$')
        ax2.set_ylabel("signed log(T_0i)")
        ax2.set_title(f"T_0i vs Gamma  (H index = {frame})")
        ax2.grid(True, which="both", alpha=0.5)
        ax2.legend(fontsize=8, ncol=2)

        return ax1, ax2

    anim = FuncAnimation(
        fig,
        update,
        frames=len(Tmats_list),
        interval=interval,
        blit=False
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
def simulate_Pii_for_gammas(
        H,
        gamma_list,
        tlist,
        double_precision=True):
    """
    Compute P_ii(t) for different Gamma values.

    For each Gamma:
    - Initialize rho(0) = |i><i| for every site i
    - Evolve rho(t)
    - Store P_ii(t) = rho_ii(t)

    Returns
    -------
    Pii_list : list of arrays
        Each entry has shape (N_sites, len(tlist))
        Pii_list[g][i, t] = P_ii(t) for site i at Gamma_g
    """

    import numpy as np
    import jax.numpy as jnp

    N = H.shape[0]
    Pii_list = []

    for Gamma in gamma_list:
        L = lindbladmatpl(H, Gamma, double_precision=double_precision)

        Pii_gamma = np.zeros((N, len(tlist)))

        for i in range(N):
            # Initial condition: excitation on site i
            rho0 = jnp.zeros((N, N), dtype=jnp.complex128).at[i, i].set(1.0)

            rhos = evolve_rho_over_time(
                L, rho0, tlist, double_precision=double_precision
            )

            # Extract P_ii(t)
            Pii_gamma[i, :] = np.real(
                np.diagonal(rhos, axis1=1, axis2=2)[:, i]
            )

        Pii_list.append(Pii_gamma)

    return Pii_list


In [ ]:
def animate_Pii_over_gamma(
        Pii_list,
        gamma_list,
        tlist,
        logy=True,
        interval=800):
    """
    Animate P_ii(t) for different Gamma values.

    Each frame corresponds to a different Gamma.
    Each curve corresponds to a different site i.
    """

    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML

    N = Pii_list[0].shape[0]

    fig, ax = plt.subplots(figsize=(8, 5))

    def update(frame):
        ax.clear()
        Pii = Pii_list[frame]

        for i in range(N):
            ax.plot(tlist, Pii[i], label=f"Site {i}")

        ax.set_xscale("log")
        if logy:
            ax.set_yscale("log")

        ax.set_xlabel("Time t (log scale)")
        ax.set_ylabel(r"$P_{ii}(t)$")
        ax.set_title(rf"Local survival $P_{{ii}}(t)$, $\Gamma={gamma_list[frame]:.2e}$")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()

        return ax

    anim = FuncAnimation(
        fig, update, frames=len(gamma_list),
        interval=interval, blit=False
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
def save_last_figure(fmt="pdf", prefix="figure", overwrite=True):
    fig = plt.gcf()
    idx = next(fig_counter)

    fname = f"{prefix}_{idx}.{fmt}"
    fpath = os.path.join(save_folder, fname)

    # Save
    fig.savefig(fpath, bbox_inches="tight", dpi=300)
    print(f"Saved: {fpath}")

In [ ]:
def save_all_figures(prefix="spin_bath", fmt="pdf"):
    fig_nums = plt.get_fignums()
    if not fig_nums:
        print("No figures to save.")
        return
    for i, fignum in enumerate(fig_nums, start=1):
        fig = plt.figure(fignum)
        fname = os.path.join(save_folder, f"{prefix}_{i}.{fmt}")
        fig.savefig(fname, bbox_inches="tight", dpi=300)
        print(f"Saved: {fname}")